# 02 — Batch Analysis

**Course:** Big Data Management Systems and Tools  
**Project:** Batch vs. Streaming Analytics at Scale (Apache Spark)  
**Dataset:** City of Chicago — Traffic Crashes – Crashes (Socrata ID: 85ca-t3if)

---

## Purpose

This notebook implements the **Batch Layer** of the project's Lambda Architecture. It reads the canonical **Silver** dataset and computes full-history, time-based aggregates that answer the project's core analytical questions.

**Strict constraints:**
- Reads **Silver Parquet only** — no raw CSV, no ETL logic (those belong to `01_bronze_to_silver.ipynb`).
- **No streaming APIs** — this notebook produces stable batch results only.
- Writes Gold outputs to `outputs/batch/`.

---

## Role in the Behavioral Comparison

This notebook serves as the **baseline reference** for the batch vs. streaming comparison:
- Demonstrates the **batch processing model**: full-dataset scan → aggregation → write.
- Records **total elapsed wall-clock time** as the batch latency metric.
- Its Gold outputs define the "complete history" results that streaming approximates incrementally.

---

## Aggregations Produced

| # | Output | Method |
|---|---|---|
| 1 | Hourly event-time window crash counts (`crashes_by_hour_window`) | DataFrame API |
| 2 | Crashes per calendar date | DataFrame API |
| 3 | Crashes per year/month | DataFrame API |
| 4 | Injury totals & fatals per month | DataFrame API |
| 5 | Contextual breakdowns (weather / lighting / roadway) | DataFrame API |

Outputs: `outputs/batch/`

In [ ]:
import os
import time
import findspark
from pyspark.sql import SparkSession

# ── Spark / Java environment ──────────────────────────────────────────────────
os.environ["JAVA_HOME"] = r"C:\Program Files\Eclipse Adoptium\jdk-11.0.30.7-hotspot"

# ── Hadoop / winutils (Windows only) ─────────────────────────────────────────
os.environ["HADOOP_HOME"] = r"C:\hadoop"
os.environ["PATH"] = r"C:\hadoop\bin" + ";" + os.environ.get("PATH", "")

findspark.init()

# ── Path constants ─────────────────────────────────────────────────────────────
NOTEBOOK_DIR  = os.path.dirname(os.path.abspath("__file__"))
PROJECT_ROOT  = os.path.join(NOTEBOOK_DIR, "..")
SILVER_PATH    = os.path.join(PROJECT_ROOT, "data", "silver", "crashes_parquet")
GOLD_BATCH_PATH = os.path.join(PROJECT_ROOT, "outputs", "batch")

print(f"Silver source  : {os.path.abspath(SILVER_PATH)}")
print(f"Gold target    : {os.path.abspath(GOLD_BATCH_PATH)}")

# ── Wall-clock timer — used as the batch latency metric ──────────────────────
batch_start = time.time()

# ── SparkSession ───────────────────────────────────────────────────────────────
spark = (
    SparkSession.builder
    .master("local[*]")
    .appName("batch_analysis")
    .config("spark.driver.memory", "12g")
    .config("spark.sql.execution.arrow.pyspark.enabled", True)
    .config("spark.sql.repl.eagerEval.enabled", True)
    .getOrCreate()
)

print(f"Spark version  : {spark.version}")
print(f"Timer started  : {time.strftime('%H:%M:%S')}")

## Step 1 — Read Silver

The Silver dataset is the **only** data source for this notebook. It was produced by `01_bronze_to_silver.ipynb` and stored as partitioned Parquet.

**Schema reminder:**

| Column | Type | Notes |
|---|---|---|
| `CRASH_RECORD_ID` | string | Traceability |
| `CRASH_DATE` | timestamp | Parsed event time |
| `INJURIES_TOTAL` | int | |
| `INJURIES_FATAL` | int | |
| `MOST_SEVERE_INJURY` | string | |
| `WEATHER_CONDITION` | string | |
| `LIGHTING_CONDITION` | string | |
| `ROADWAY_SURFACE_COND` | string | |
| `PRIM_CONTRIBUTORY_CAUSE` | string | |
| `SEC_CONTRIBUTORY_CAUSE` | string | |
| `year` | int | Partition key |
| `month` | int | Partition key |

Spark reads only the partitions needed for a given query (partition pruning). Reads that filter by `year` or `month` will not scan unrelated files.

In [ ]:
silver_df = spark.read.parquet(SILVER_PATH)

print("=== Silver Schema ===")
silver_df.printSchema()

silver_count = silver_df.count()
print(f"Total Silver rows : {silver_count:,}")

## Step 2 — Feature Derivation

All batch aggregations are time-based. We derive additional time-bucket columns from `CRASH_DATE` once, then **cache** the enriched DataFrame.

**New columns:**

| Column | Expression | Purpose |
|---|---|---|
| `day_of_week` | `F.date_format("CRASH_DATE", "EEEE")` | Full name (Monday … Sunday) |
| `date` | `F.to_date("CRASH_DATE")` | Calendar date (YYYY-MM-DD) for daily rollup |
| `week_of_year` | `F.weekofyear("CRASH_DATE")` | ISO week number for weekly rollup |

**Why cache?** All 5 aggregation queries below scan this same DataFrame. Calling `.cache()` pins it in Spark's memory after the first scan, so subsequent queries skip re-reading from disk. This is the standard batch optimization for repeated reads over the same dataset.

In [ ]:
from pyspark.sql import functions as F

analysis_df = (
    silver_df
    .withColumn("day_of_week",  F.date_format("CRASH_DATE", "EEEE"))
    .withColumn("date",         F.to_date("CRASH_DATE"))
    .withColumn("week_of_year", F.weekofyear("CRASH_DATE"))
)

# Cache — all aggregations below read from this DataFrame
analysis_df.cache()

# Trigger the cache by counting (forces materialization)
print(f"analysis_df cached with {analysis_df.count():,} rows")
print(f"Columns : {analysis_df.columns}")

## Step 3 — Batch Aggregations

The five aggregations below cover the project's core analytical questions. Each uses the cached `analysis_df` and is stored as a named DataFrame for later writing to Gold.

---

### Aggregation 1 — Hourly Event-Time Windows (`crashes_by_hour_window`)

Groups all crashes into fixed, non-overlapping 1-hour buckets using `CRASH_DATE` as the event-time column — the **same window definition** that `streaming_job.py` will use. This makes the batch output the direct reference baseline for the streaming comparison: batch computes all windows in a single full-dataset scan; streaming produces them incrementally as files arrive.

In [ ]:
# Agg 1 — Fixed hourly event-time windows using F.window()
# window("CRASH_DATE", "1 hour") creates non-overlapping 1-hour buckets anchored
# to wall-clock time. This is the SAME window definition used in streaming_job.py,
# making the batch vs. streaming output comparison semantically consistent.
crashes_by_hour_window = (
    analysis_df
    .groupBy(F.window("CRASH_DATE", "1 hour"))
    .agg(
        F.count("*").alias("crashes"),
        F.sum("INJURIES_TOTAL").alias("injuries_total"),
        F.sum("INJURIES_FATAL").alias("injuries_fatal"),
    )
    .withColumn("window_start", F.col("window.start"))
    .withColumn("window_end",   F.col("window.end"))
    .drop("window")
    .orderBy("window_start")
)

total_windows = crashes_by_hour_window.count()
print(f"Total hourly windows : {total_windows:,}")
crashes_by_hour_window.show(10, truncate=False)

### Aggregation 2 — Crashes per Calendar Date

A daily rollup over the full 2022–2025 period. This produces a time series of crash counts that can reveal seasonal patterns, long-term trends, and anomalous days. In batch, this requires scanning the full dataset; streaming would produce this incrementally as each day's data arrives.

In [ ]:
# Agg 2 — Crashes per calendar date
crashes_by_date = (
    analysis_df
    .groupBy("date")
    .count()
    .withColumnRenamed("count", "crash_count")
    .orderBy("date")
)

total_days = crashes_by_date.count()
print(f"Distinct days in dataset : {total_days:,}")
crashes_by_date.show(10, truncate=False)

### Aggregation 3 — Crashes per Year/Month

A monthly rollup using the `year` and `month` **partition columns** already present in Silver. Filtering or grouping by these columns allows Spark to apply **partition pruning** — skipping unrelated files entirely. This is a key advantage of the partitioned Parquet layout established in `01_bronze_to_silver.ipynb`.

In [ ]:
# Agg 3 — Crashes per year/month (leverages Silver partition columns)
crashes_by_month = (
    analysis_df
    .groupBy("year", "month")
    .count()
    .withColumnRenamed("count", "crash_count")
    .orderBy("year", "month")
)
crashes_by_month.show(48, truncate=False)

### Aggregation 4 — Injury Totals and Fatal Injuries per Month

Computes `SUM(INJURIES_TOTAL)` and `SUM(INJURIES_FATAL)` grouped by year and month. This is the primary **severity trend** metric — it shows not just how many crashes occurred, but how harmful they were over time. Nulls in the integer columns (from unparseable source values) are treated as zero by `SUM`, which is the correct semantic here.

In [ ]:
# Agg 4 — Total and fatal injuries per year/month
injuries_by_month = (
    analysis_df
    .groupBy("year", "month")
    .agg(
        F.sum("INJURIES_TOTAL").alias("total_injuries"),
        F.sum("INJURIES_FATAL").alias("fatal_injuries"),
        F.count("*").alias("crash_count"),
    )
    .orderBy("year", "month")
)
injuries_by_month.show(48, truncate=False)

### Aggregation 5 — Contextual Breakdowns (Weather / Lighting / Roadway)

Three separate breakdowns by the environmental context columns retained in Silver. These give the project a second analytical dimension beyond pure time — important for a complete "management-style" report. Each is a simple `GROUP BY` + `COUNT`, sorted by crash frequency.

In [ ]:
def context_breakdown(col_name):
    """Count crashes grouped by a categorical context column, sorted descending."""
    return (
        analysis_df
        .groupBy(col_name)
        .count()
        .withColumnRenamed("count", "crash_count")
        .orderBy(F.col("crash_count").desc())
    )

crashes_by_weather  = context_breakdown("WEATHER_CONDITION")
crashes_by_lighting = context_breakdown("LIGHTING_CONDITION")
crashes_by_roadway  = context_breakdown("ROADWAY_SURFACE_COND")

print("=== Weather Condition ===")
crashes_by_weather.show(truncate=False)

print("=== Lighting Condition ===")
crashes_by_lighting.show(truncate=False)

print("=== Roadway Surface Condition ===")
crashes_by_roadway.show(truncate=False)

## Step 4 — Visualizations

We convert **aggregated** (small) DataFrames to pandas for visualization — never the full Silver dataset.

Two charts:
1. **Hourly event-time windows (January 2022 sample)** — line chart of crash counts per hourly window; demonstrates the window-based output that streaming will mirror.
2. **Monthly crash count 2022–2025** — bar chart showing the full time series at month granularity.

These are report-ready figures that illustrate the batch layer's output clearly.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

# ── Chart 1: Hourly event-time windows — January 2022 sample ─────────────────
# We sample one month so the chart stays readable (~744 hourly data points).
window_pd = crashes_by_hour_window.toPandas()
sample = window_pd[
    (window_pd["window_start"] >= "2022-01-01") &
    (window_pd["window_start"] <  "2022-02-01")
].sort_values("window_start")

fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(sample["window_start"], sample["crashes"],
        color="steelblue", linewidth=0.8)
ax.fill_between(sample["window_start"], sample["crashes"],
                alpha=0.2, color="steelblue")
ax.set_title("Hourly Crash Counts — January 2022 (Event-Time Windows, 1h)",
             fontsize=13, fontweight="bold")
ax.set_xlabel("Window Start")
ax.set_ylabel("Crashes per Hour")
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{int(x):,}"))
plt.tight_layout()
plt.show()

# ── Chart 2: Monthly Crash Count 2022–2025 ───────────────────────────────────
month_pd = crashes_by_month.toPandas()
month_pd["period"] = (month_pd["year"].astype(str) + "-" +
                      month_pd["month"].astype(str).str.zfill(2))

fig, ax = plt.subplots(figsize=(16, 4))
ax.bar(month_pd["period"], month_pd["crash_count"],
       color="darkorange", edgecolor="white")
ax.set_title("Monthly Crash Count (2022–2025)", fontsize=14, fontweight="bold")
ax.set_xlabel("Year-Month")
ax.set_ylabel("Number of Crashes")
ax.tick_params(axis="x", rotation=70, labelsize=8)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{int(x):,}"))
plt.tight_layout()
plt.show()

## Step 5 — Write Gold Outputs

Each aggregation is written as a separate Parquet dataset under `outputs/batch/`. This constitutes the **Gold layer** for the batch processing path.

**Design choices:**
- `mode("overwrite")` makes re-runs idempotent.
- One folder per aggregation — separates concerns and allows downstream tools to read only what they need.
- Parquet format is consistent with Silver, keeping the pipeline uniform.

In [ ]:
def write_gold(df, name):
    """Write a batch aggregation result to the Gold output path."""
    path = os.path.join(GOLD_BATCH_PATH, name)
    df.write.mode("overwrite").parquet(path)
    print(f"  ✓ Written: outputs/batch/{name}/")

print(f"Writing Gold outputs to: {os.path.abspath(GOLD_BATCH_PATH)}\n")

write_gold(crashes_by_hour_window, "crashes_by_hour_window")
write_gold(crashes_by_date,        "crashes_by_date")
write_gold(crashes_by_month,       "crashes_by_month")
write_gold(injuries_by_month,      "injuries_by_month")
write_gold(crashes_by_weather,     "crashes_by_weather")
write_gold(crashes_by_lighting,    "crashes_by_lighting")
write_gold(crashes_by_roadway,     "crashes_by_roadway")

# ── Stop wall-clock timer ─────────────────────────────────────────────────────
batch_elapsed = time.time() - batch_start
print(f"\nAll Gold outputs written.")

## Step 6 — Batch Latency Baseline

The total elapsed wall-clock time from `SparkSession` start to the last Gold write is the **batch latency metric** used in the behavioral comparison.

In the batch model, **no results are available until all data has been processed and written**. This is the fundamental latency characteristic of batch: high throughput, complete accuracy, but results arrive only after the full job completes.

This will be directly contrasted with the streaming model, where partial (windowed) results appear incrementally as each file arrives.

In [ ]:
# ================================
# BATCH RUN SUMMARY (refined output)
# ================================
from pyspark.sql import functions as F

# 1. Time range covered (min/max CRASH_DATE)
min_date = analysis_df.agg(F.min('CRASH_DATE')).first()[0]
max_date = analysis_df.agg(F.max('CRASH_DATE')).first()[0]

# 2. Event-time column and window definition
event_time_col = 'CRASH_DATE'
window_def = '1 hour (event-time, fixed, non-overlapping)'

# 3. Aggregation and output counts
core_aggregations = 5
gold_outputs = 7
gold_output_names = [
    'crashes_by_hour_window',
    'crashes_by_date',
    'crashes_by_month',
    'injuries_by_month',
    'crashes_by_weather',
    'crashes_by_lighting',
    'crashes_by_roadway',
]

# 4. Input format and partitioning (optional, if available)
input_format = 'Parquet (partitioned by year, month)'

# 5. Batch latency (end-to-end job runtime)
minutes, seconds = divmod(batch_elapsed, 60)

print("=" * 55)
print("  BATCH RUN SUMMARY (02_batch_analysis.ipynb)")
print("=" * 55)
print(f"  Dataset             : Chicago Traffic Crashes")
print(f"  Time range covered  : {min_date.date()} to {max_date.date()}")
print(f"  Rows processed      : {silver_count:,}")
print(f"  Event-time column   : {event_time_col}")
print(f"  Hourly window size  : {window_def}")
print(f"  Core aggregation queries: {core_aggregations}")
print(f"  Gold outputs written    : {gold_outputs}")
print(f"  Gold outputs:")
for name in gold_output_names:
    print(f"    - {name}")
print(f"  Total elapsed time  : {int(minutes)}m {seconds:.1f}s")
print(f"  Gold output path    : outputs/batch/")
print(f"  Input format        : {input_format}")
print("=" * 55)
print()
print("Latency note: In the batch model, end-to-end latency is defined as the total job runtime; no partial results are available before job completion.")
print("In contrast, the streaming model will be evaluated based on time‑to‑first‑result and incremental updates per micro‑batch.")

In [ ]:
# Release the cached DataFrame and stop Spark cleanly.
analysis_df.unpersist()
spark.stop()
print("Spark session stopped. Batch analysis complete.")